In [14]:
import pandas as pd

hip_train_path = "/kaggle/input/datasets/mkhaleddeaf/hip-data-for-humanization/ai_detection_project/data/hip/hip_train.jsonl"

df_hip = pd.read_json(hip_train_path, lines=True)
print("Shape:", df_hip.shape)
print("Columns:", list(df_hip.columns))
print(df_hip.head(2))

candidate_cols = [c for c in df_hip.columns if 'model' in c.lower() or 'source' in c.lower()]
print("\nCandidate model/source columns:", candidate_cols)
for c in candidate_cols:
    print(f"\n{c} value counts:")
    print(df_hip[c].value_counts())
    

Shape: (8902, 6)
Columns: ['group_id', 'dataset', 'source', 'ai_text', 'human_text', 'gpt5_nano_semantic_score']
       group_id dataset      source  \
0  7878b22029c7    mage  xsum_human   
1  c44c9f05780f    mage  xsum_human   

                                             ai_text  \
0  In a text to authorities, Donald “Chip” Pugh a...   
1  At 24, Moses has not featured competitively fo...   

                                          human_text  gpt5_nano_semantic_score  
0  Donald "Chip" Pugh texted police a photo of hi...                        10  
1  Moses, 24, has not played a competitive game f...                        10  

Candidate model/source columns: ['source']

source value counts:
source
xsum_human     1552
squad_human    1549
tldr_human     1344
abstracts      1318
wiki           1303
books           919
news            847
cnn_human        70
Name: count, dtype: int64


In [15]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    if 'ai-humanizer' in root.lower() or any('ai_vs_humanized' in f.lower() or ('train' in f.lower() and f.endswith('.csv')) for f in files):
        print(root)
        for f in files:
            print("   -", f)

/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw
   - README.md
   - main.png
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git
   - config
   - packed-refs
   - HEAD
   - index
   - description
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git/info
   - exclude
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git/refs
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git/refs/heads
   - main
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git/refs/remotes
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git/refs/remotes/origin
   - HEAD
/kaggle/input/datasets/mkhaleddeaf/ai-humanizer-dataset/ai_humanized_raw/.git/hooks
   - pre-merge-commit.sample
   - prepare-commit-msg.sample
   - update.sample
   - pre-push.sample
   - pre-rebase.sample
   - pre-appl

In [17]:
import os, glob
import pandas as pd

def find_file(filename, search_root='/kaggle/input'):
    matches = glob.glob(os.path.join(search_root, '**', filename), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {search_root}")
    return matches[0]

hip_train_path = find_file('hip_train.jsonl')
hip_val_path = find_file('hip_val.jsonl')
hip_test_path = find_file('hip_test.jsonl')

hip_train = pd.read_json(hip_train_path, lines=True)
hip_val = pd.read_json(hip_val_path, lines=True)
hip_test = pd.read_json(hip_test_path, lines=True)

ai_hum_candidates = glob.glob('/kaggle/input/**/ai_vs_humanized/train.csv', recursive=True)
ai_hum_dir = os.path.dirname(ai_hum_candidates[0])

ai_hum_train = pd.read_json(os.path.join(ai_hum_dir, 'train.csv'), lines=True)
ai_hum_val = pd.read_json(os.path.join(ai_hum_dir, 'val.csv'), lines=True)
ai_hum_test = pd.read_json(os.path.join(ai_hum_dir, 'test.csv'), lines=True)

print("HIP:", hip_train.shape, hip_val.shape, hip_test.shape)
print("AI-Humanizer:", ai_hum_train.shape, ai_hum_val.shape, ai_hum_test.shape)

HIP: (8902, 6) (785, 6) (785, 6)
AI-Humanizer: (1695, 8) (150, 8) (154, 8)


In [18]:
def unify_hip(df):
    out = pd.DataFrame()
    out['doc_id'] = df['group_id']
    out['dataset_origin'] = 'HIP'
    out['ai_text'] = df['ai_text']
    out['human_text'] = df['human_text']
    out['orig_source'] = df['source']
    return out

def unify_ai_humanizer(df):
    out = pd.DataFrame()
    out['doc_id'] = df['doc_id']
    out['dataset_origin'] = 'AI_Humanizer'
    out['ai_text'] = df['ai_text']
    out['human_text'] = df['human_text']
    out['orig_source'] = df['source']
    return out

train_combined = pd.concat([unify_hip(hip_train), unify_ai_humanizer(ai_hum_train)], ignore_index=True)
val_combined = pd.concat([unify_hip(hip_val), unify_ai_humanizer(ai_hum_val)], ignore_index=True)
test_combined = pd.concat([unify_hip(hip_test), unify_ai_humanizer(ai_hum_test)], ignore_index=True)

print("Combined train:", train_combined.shape)
print("Combined val:", val_combined.shape)
print("Combined test:", test_combined.shape)

print("\nDuplicate doc_ids across all splits combined:",
      pd.concat([train_combined['doc_id'], val_combined['doc_id'], test_combined['doc_id']]).duplicated().sum())

print("\ndataset_origin distribution per split:")
for name, df in [('train', train_combined), ('val', val_combined), ('test', test_combined)]:
    print(name, ":", df['dataset_origin'].value_counts().to_dict())

Combined train: (10597, 5)
Combined val: (935, 5)
Combined test: (939, 5)

Duplicate doc_ids across all splits combined: 1776

dataset_origin distribution per split:
train : {'HIP': 8902, 'AI_Humanizer': 1695}
val : {'HIP': 785, 'AI_Humanizer': 150}
test : {'HIP': 785, 'AI_Humanizer': 154}


In [19]:
train_combined['ai_word_count'] = train_combined['ai_text'].str.split().str.len()
train_combined['human_word_count'] = train_combined['human_text'].str.split().str.len()

print(train_combined[['ai_word_count', 'human_word_count']].describe())
print("\n95th percentile ai_word_count:", train_combined['ai_word_count'].quantile(0.95))
print("95th percentile human_word_count:", train_combined['human_word_count'].quantile(0.95))

       ai_word_count  human_word_count
count   10597.000000      10597.000000
mean      180.816552        202.691705
std        61.336149         71.406241
min        57.000000        100.000000
25%       130.000000        143.000000
50%       176.000000        195.000000
75%       226.000000        250.000000
max       775.000000        400.000000

95th percentile ai_word_count: 286.0
95th percentile human_word_count: 341.0


# **tokenizer**

In [21]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

MODEL_NAME = "t5-small"
MAX_LENGTH = 512

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

print("Tokenizer and model loaded:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)

# Quick sanity check on tokenization length for a sample row
sample_text = train_combined.iloc[0]['ai_text']
tokens = tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH)
print("\nSample input token count:", len(tokens['input_ids']))

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizer and model loaded: t5-small
Vocab size: 32100

Sample input token count: 213


In [22]:
import torch
from torch.utils.data import Dataset

PREFIX = "humanize: "
MAX_LENGTH = 512

class HumanizerDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH):
        self.ai_texts = df['ai_text'].tolist()
        self.human_texts = df['human_text'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.ai_texts)

    def __getitem__(self, idx):
        input_text = PREFIX + self.ai_texts[idx]
        target_text = self.human_texts[idx]

        model_inputs = self.tokenizer(
            input_text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        labels = self.tokenizer(
            target_text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        label_ids = labels["input_ids"].squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels": label_ids
        }

train_dataset = HumanizerDataset(train_combined, tokenizer)
val_dataset = HumanizerDataset(val_combined, tokenizer)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

sample = train_dataset[0]
print("\nSample shapes:")
print("input_ids:", sample["input_ids"].shape)
print("attention_mask:", sample["attention_mask"].shape)
print("labels:", sample["labels"].shape)
print("\nDecoded input:", tokenizer.decode(sample["input_ids"], skip_special_tokens=True)[:150])

Train dataset size: 10597
Val dataset size: 935

Sample shapes:
input_ids: torch.Size([512])
attention_mask: torch.Size([512])
labels: torch.Size([512])

Decoded input: humanize: In a text to authorities, Donald “Chip” Pugh attached a selfie, remarking that the image “here is a better photo, that one is terrible.” Lim


In [23]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/humanizer_t5_small",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer ready.")
print("Total training steps per epoch:", len(train_dataset) // training_args.per_device_train_batch_size)

Trainer ready.
Total training steps per epoch: 1324


In [24]:
train_result = trainer.train()

print("Training completed.")
print(train_result.metrics)

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,4.061801,3.619333
2,3.818708,3.525438
3,3.685104,3.498806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training completed.
{'train_runtime': 1759.3133, 'train_samples_per_second': 18.07, 'train_steps_per_second': 1.131, 'total_flos': 4302651210596352.0, 'train_loss': 3.9671448659153667, 'epoch': 3.0}


In [25]:
model.eval()
device = next(model.parameters()).device

sample_rows = val_combined.sample(3, random_state=42)

for idx, row in sample_rows.iterrows():
    input_text = PREFIX + row['ai_text']
    input_ids = tokenizer(input_text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).input_ids.to(device)

    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("=" * 80)
    print("ORIGINAL AI TEXT (first 200 chars):")
    print(row['ai_text'][:200])
    print("\nGENERATED (humanized) OUTPUT:")
    print(generated[:400])
    print("\nREFERENCE HUMAN TEXT (first 200 chars):")
    print(row['human_text'][:200])

ORIGINAL AI TEXT (first 200 chars):
Set during the early 1700s across France and Louisiana, the tale centers on Chevalier des Grieux and his beloved Manon Lescaut. Though Des Grieux hails from a noble, landed lineage, he forfeits his an

GENERATED (humanized) OUTPUT:
Chevalier des Grieux and his beloved Manon Lescaut are set in the early 1700s across France and Louisiana. Although Des Grieux is a noble, landed man, he forfeits his ancestral fortune and disappoints his father by eloping with Manon. In Paris, the couple enjoys a carefree domestic life, while Des Grieux struggles to indulge Manon's desire for luxury, financing their needs by borrowing from his st

REFERENCE HUMAN TEXT (first 200 chars):
 Set in France and Louisiana in the early 18th century, the story follows the hero, the Chevalier des Grieux, and his lover, Manon Lescaut. Des Grieux comes from a noble and landed family, but forfeit
ORIGINAL AI TEXT (first 200 chars):
Semantic segmentation remains one of the most demandi

In [27]:
trainer.save_model("/kaggle/working/humanizer_t5_small_final")
tokenizer.save_pretrained("/kaggle/working/humanizer_t5_small_final")
print("Model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


In [29]:
!pip install rouge_score sacrebleu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.1 MB/s eta 0:00:00


In [31]:
import os

CHECKPOINT_DIR = "/kaggle/working/humanizer_t5_small_final"
OUTPUT_DIR = "/kaggle/working/humanizer_t5_small"

if 'trainer' not in dir():
    print("Session was restarted or trainer not in memory. Reloading model from saved checkpoint.")
    from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

    tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT_DIR)
    model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_DIR)

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100
    )
else:
    print("Reusing model and trainer already in memory.")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=8,
    predict_with_generate=True,
    generation_max_length=512,
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

resume_ckpt = os.path.exists(OUTPUT_DIR) and any('checkpoint' in d for d in os.listdir(OUTPUT_DIR)) if os.path.exists(OUTPUT_DIR) else False

train_result = trainer.train(resume_from_checkpoint=resume_ckpt if resume_ckpt else None)

print("Training completed.")
print(train_result.metrics)

trainer.save_model("/kaggle/working/humanizer_t5_small_final_v2")
tokenizer.save_pretrained("/kaggle/working/humanizer_t5_small_final_v2")
print("Model saved to humanizer_t5_small_final_v2")

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Reusing model and trainer already in memory.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
4,3.718512,3.464722
5,3.588492,3.435074
6,3.633244,3.405823
7,3.530508,3.404373
8,3.495161,3.396114


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training completed.
{'train_runtime': 2945.4402, 'train_samples_per_second': 28.782, 'train_steps_per_second': 1.801, 'total_flos': 1.1473736561590272e+16, 'train_loss': 2.2476236967659284, 'epoch': 8.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to humanizer_t5_small_final_v2


In [33]:
model.eval()
device = next(model.parameters()).device

sample_rows = val_combined.sample(3, random_state=42)

for idx, row in sample_rows.iterrows():
    input_text = PREFIX + row['ai_text']
    input_ids = tokenizer(input_text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).input_ids.to(device)

    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("=" * 80)
    print("ORIGINAL AI TEXT (first 200 chars):")
    print(row['ai_text'][:200])
    print("\nGENERATED (humanized) OUTPUT:")
    print(generated[:400])
    print("\nREFERENCE HUMAN TEXT (first 200 chars):")
    print(row['human_text'][:200])

ORIGINAL AI TEXT (first 200 chars):
Set during the early 1700s across France and Louisiana, the tale centers on Chevalier des Grieux and his beloved Manon Lescaut. Though Des Grieux hails from a noble, landed lineage, he forfeits his an

GENERATED (humanized) OUTPUT:
The novel is set in the early 1700s in France and Louisiana, based on Chevalier des Grieux and his beloved Manon Lescaut. Although Des Grieux is from a noble, landed family, he forfeits his ancestral fortune and disappoints his father by eloping with Manon. In Paris, the couple enjoy a carefree domestic life, while Des Grieux struggles to indulge Manon's desire for luxury, financing their needs by

REFERENCE HUMAN TEXT (first 200 chars):
 Set in France and Louisiana in the early 18th century, the story follows the hero, the Chevalier des Grieux, and his lover, Manon Lescaut. Des Grieux comes from a noble and landed family, but forfeit
ORIGINAL AI TEXT (first 200 chars):
Semantic segmentation remains one of the most demandi

In [32]:
from rouge_score import rouge_scorer
import sacrebleu
import torch

model.eval()
device = next(model.parameters()).device
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def generate_batch(texts, batch_size=16):
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = [PREFIX + t for t in batch]
        enc = tokenizer(inputs, return_tensors="pt", max_length=MAX_LENGTH,
                         truncation=True, padding=True).to(device)
        with torch.no_grad():
            out_ids = model.generate(**enc, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        outputs.extend(decoded)
    return outputs

# Use a subset of test set for speed (full set if small enough)
eval_df = test_combined.sample(min(200, len(test_combined)), random_state=42).reset_index(drop=True)

generated_texts = generate_batch(eval_df['ai_text'].tolist())

rouge1_gen, rouge2_gen, rougeL_gen = [], [], []
rouge1_base, rouge2_base, rougeL_base = [], [], []

for i, row in eval_df.iterrows():
    ref = row['human_text']
    gen = generated_texts[i]
    baseline = row['ai_text']

    s_gen = scorer.score(ref, gen)
    s_base = scorer.score(ref, baseline)

    rouge1_gen.append(s_gen['rouge1'].fmeasure)
    rouge2_gen.append(s_gen['rouge2'].fmeasure)
    rougeL_gen.append(s_gen['rougeL'].fmeasure)

    rouge1_base.append(s_base['rouge1'].fmeasure)
    rouge2_base.append(s_base['rouge2'].fmeasure)
    rougeL_base.append(s_base['rougeL'].fmeasure)

bleu_gen = sacrebleu.corpus_bleu(generated_texts, [eval_df['human_text'].tolist()])
bleu_base = sacrebleu.corpus_bleu(eval_df['ai_text'].tolist(), [eval_df['human_text'].tolist()])

print(f"Evaluated on {len(eval_df)} test samples\n")
print("=" * 60)
print("MODEL OUTPUT vs HUMAN REFERENCE:")
print(f"  ROUGE-1: {sum(rouge1_gen)/len(rouge1_gen):.4f}")
print(f"  ROUGE-2: {sum(rouge2_gen)/len(rouge2_gen):.4f}")
print(f"  ROUGE-L: {sum(rougeL_gen)/len(rougeL_gen):.4f}")
print(f"  BLEU: {bleu_gen.score:.2f}")

print("\n" + "=" * 60)
print("BASELINE (original AI text, no humanization) vs HUMAN REFERENCE:")
print(f"  ROUGE-1: {sum(rouge1_base)/len(rouge1_base):.4f}")
print(f"  ROUGE-2: {sum(rouge2_base)/len(rouge2_base):.4f}")
print(f"  ROUGE-L: {sum(rougeL_base)/len(rougeL_base):.4f}")
print(f"  BLEU: {bleu_base.score:.2f}")

print("\n" + "=" * 60)
print("DELTA (positive = model improved over baseline):")
print(f"  ROUGE-1: {sum(rouge1_gen)/len(rouge1_gen) - sum(rouge1_base)/len(rouge1_base):+.4f}")
print(f"  ROUGE-L: {sum(rougeL_gen)/len(rougeL_gen) - sum(rougeL_base)/len(rougeL_base):+.4f}")
print(f"  BLEU: {bleu_gen.score - bleu_base.score:+.2f}")

Evaluated on 200 test samples

MODEL OUTPUT vs HUMAN REFERENCE:
  ROUGE-1: 0.6764
  ROUGE-2: 0.3708
  ROUGE-L: 0.5498
  BLEU: 27.37

BASELINE (original AI text, no humanization) vs HUMAN REFERENCE:
  ROUGE-1: 0.6541
  ROUGE-2: 0.3419
  ROUGE-L: 0.5281
  BLEU: 23.51

DELTA (positive = model improved over baseline):
  ROUGE-1: +0.0222
  ROUGE-L: +0.0218
  BLEU: +3.86


In [34]:
import os
import torch

MODEL_DIR = "/kaggle/working/humanizer_t5_small_final"

os.makedirs(MODEL_DIR, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Save training metadata
metadata = {
    "model": "t5-small",
    "task": "AI text humanization",
    "epochs": 8,
    "max_length": MAX_LENGTH,
    "prefix": PREFIX,
}

import json

with open(os.path.join(MODEL_DIR, "training_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Model saved to:", MODEL_DIR)
print("Files:")
for file in os.listdir(MODEL_DIR):
    print(" -", file)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/humanizer_t5_small_final
Files:
 - generation_config.json
 - training_metadata.json
 - config.json
 - tokenizer_config.json
 - model.safetensors
 - tokenizer.json
 - training_args.bin


In [35]:
import os

print("Directory exists:", os.path.exists(MODEL_DIR))

for root, dirs, files in os.walk(MODEL_DIR):
    for file in files:
        path = os.path.join(root, file)
        print(f"{file}: {os.path.getsize(path) / (1024**2):.2f} MB")

Directory exists: True
generation_config.json: 0.00 MB
training_metadata.json: 0.00 MB
config.json: 0.00 MB
tokenizer_config.json: 0.00 MB
model.safetensors: 230.83 MB
tokenizer.json: 2.01 MB
training_args.bin: 0.01 MB


In [36]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

TEST_MODEL_DIR = "/kaggle/working/humanizer_t5_small_final"

test_tokenizer = T5Tokenizer.from_pretrained(TEST_MODEL_DIR)
test_model = T5ForConditionalGeneration.from_pretrained(TEST_MODEL_DIR)

test_model.eval()

print("Model loaded successfully!")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded successfully!


In [37]:
test_text = (
    "Artificial intelligence has significantly transformed "
    "many industries in recent years."
)

inputs = test_tokenizer(
    PREFIX + test_text,
    return_tensors="pt",
    max_length=MAX_LENGTH,
    truncation=True
)

with torch.no_grad():
    output_ids = test_model.generate(
        **inputs,
        max_length=MAX_LENGTH,
        num_beams=4,
        early_stopping=True
    )

output = test_tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("INPUT:")
print(test_text)

print("\nOUTPUT:")
print(output)

INPUT:
Artificial intelligence has significantly transformed many industries in recent years.

OUTPUT:
Artificial intelligence has changed many industries in the last few years.


In [38]:
import shutil
import os

ZIP_BASE = "/kaggle/working/humanizer_t5_small_8epochs"

zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    MODEL_DIR
)

print("ZIP created:")
print(zip_path)

print(
    f"Size: {os.path.getsize(zip_path) / (1024**2):.2f} MB"
)

ZIP created:
/kaggle/working/humanizer_t5_small_8epochs.zip
Size: 214.54 MB
